# 07 – Truy xuất và đánh giá

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT))

INDEX_PATH = PROJECT_ROOT / "knowledge" / "index" / "faiss.index"
META_PATH = PROJECT_ROOT / "knowledge" / "index" / "chunks_meta.json"
CHUNKS_PATH = PROJECT_ROOT / "knowledge" / "chunks" / "chunks.jsonl"
VACCINES_PATH = PROJECT_ROOT / "data" / "mock" / "vaccines_from_db.json"

print("index exists", INDEX_PATH.exists(), "meta", META_PATH.exists())
assert INDEX_PATH.exists(), "Chạy notebook 06_build_index trước"


index exists True meta True


In [2]:
from app.chatbot.rag.embedder import load_embedder
from app.chatbot.rag.vector_store import VectorStore
from app.chatbot.rag.retriever import Retriever
from app.core.config import settings

metas = json.loads(META_PATH.read_text(encoding="utf-8"))
texts = [m.get("text") or "" for m in metas]

embedder = load_embedder(prefer_bge=True, corpus_for_tfidf=texts)
store = VectorStore.load(INDEX_PATH, META_PATH)
print("store", store.size, "dim", store.dim, "embedder", embedder.model_name)

bm25 = None
try:
    from app.chatbot.rag.bm25 import BM25Store
    bm25 = BM25Store(texts, metas=metas)
    print("BM25 ready")
except Exception as e:
    print("BM25 skip:", e)

retriever = Retriever(
    embedder=embedder,
    store=store,
    bm25=bm25,
    top_k=settings.retrieval_top_k,
    min_score=settings.retrieval_min_score,
    score_margin=getattr(settings, "retrieval_score_margin", 0.08),
)
print("min_score", retriever.min_score, "margin", retriever.score_margin)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

store 100 dim 1024 embedder BAAI/bge-m3
BM25 ready
min_score 0.48 margin 0.08


## Thử vài câu (có / không entity vaccine)

In [3]:
from app.chatbot.entities import EntityExtractor

extractor = EntityExtractor(vaccines_path=VACCINES_PATH) if VACCINES_PATH.exists() else EntityExtractor()

samples = [
    "Tôi 20 tuổi nên tiêm vaccine nào?",
    "Vaccine HPV phòng bệnh gì?",
    "Tôi đã tiêm mũi 1 MMR rồi, tiếp theo?",
    "Khám sàng lọc trước tiêm trẻ em",
    "Chống chỉ định tiêm chủng",
]

for q in samples:
    ent = extractor.extract(q)
    vid = ent["vaccine_ids"][0] if ent.get("vaccine_ids") else None
    result = retriever.retrieve(q, vaccine_id=vid)
    print("\n===", q)
    print("entities:", {k: ent.get(k) for k in ("vaccine_ids", "age_years", "dose_number")})
    print("enough=", result["enough_evidence"], "max=", round(result["max_score"], 3),
          "raw_max=", round(result.get("raw_max_score") or 0, 3),
          "reason=", result.get("rejected_reason"))
    for h in result["hits"][:3]:
        print(f"  [{h['score']:.3f}] {h.get('chunk_id')} | {(h.get('text') or '')[:80]}")



=== Tôi 20 tuổi nên tiêm vaccine nào?
entities: {'vaccine_ids': [], 'age_years': 20.0, 'dose_number': None}
enough= True max= 0.7 raw_max= 0.7 reason= None
  [0.700] 6381586785791217441575-qđ-byt-về-việc-hướng-dẫn-khám-sàng-lọc-trước-tiêm-chủng-đối-với-trẻ-em_0045 | n thiên phổ biến
(CVID)
- Suy giảm miễn dịch không
có Gammaglobulin
- Vắc xin số
  [0.660] tt-10-2024_0005 | n
Haemophilus
influenzae týp b đơn
giá hoặc vắc xin phối
hợp có chứa thành
phần 

=== Vaccine HPV phòng bệnh gì?
entities: {'vaccine_ids': [11], 'age_years': None, 'dose_number': None}
enough= True max= 1.0 raw_max= 1.0 reason= None
  [1.000] db-vaccine_11 | Vaccine: Vắc xin HPV (Human Papillomavirus)
Nhà sản xuất: MSD – Hoa Kỳ
Phòng bện

=== Tôi đã tiêm mũi 1 MMR rồi, tiếp theo?
entities: {'vaccine_ids': [6], 'age_years': None, 'dose_number': 1}
enough= True max= 0.855 raw_max= 0.855 reason= None
  [0.855] 6381586785791217441575-qđ-byt-về-việc-hướng-dẫn-khám-sàng-lọc-trước-tiêm-chủng-đối-với-trẻ-em_0031 |  bệnh nhi

## Eval nhanh: positive + negative

- **Positive:** kỳ vọng `enough_evidence=True` và hit chứa keyword.
- **Negative:** kỳ vọng `enough_evidence=False` → góp phần `neg_ok_rate > 0`.


In [4]:
GOLD_POS = [
    {"query": "Khám sàng lọc trước tiêm chủng", "must_contain": ["sàng lọc", "1575", "scr"]},
    {"query": "Chống chỉ định tiêm", "must_contain": ["chống chỉ định", "sàng lọc", "chong_chi"]},
    {"query": "Vaccine HPV phòng bệnh gì", "must_contain": ["hpv", "HPV"]},
    {"query": "Trẻ sơ sinh tiêm TCMR", "must_contain": ["tcmr", "sơ sinh", "TCMR"]},
]

GOLD_NEG = [
    "Thời tiết Đà Nẵng ngày mai",
    "Công thức nấu bún chả",
    "Tỷ số bóng đá hôm qua",
    "Cách học tiếng Anh giao tiếp",
    "Giá vàng SJC",
]

def hit_matches(h, keys):
    blob = (str(h.get("doc_id", "")) + " " + str(h.get("chunk_id", "")) + " " + str(h.get("text", ""))).lower()
    return any(k.lower() in blob for k in keys)

k = 5
pos_hits = 0
for g in GOLD_POS:
    r = retriever.retrieve(g["query"], top_k=k)
    ok = r["enough_evidence"] and any(hit_matches(h, g["must_contain"]) for h in r["hits"])
    pos_hits += int(ok)
    print(("OK" if ok else "MISS"), g["query"], "enough=", r["enough_evidence"], "n_hits=", len(r["hits"]))

neg_ok = 0
for q in GOLD_NEG:
    r = retriever.retrieve(q, top_k=k)
    ok = not r["enough_evidence"]
    neg_ok += int(ok)
    print(("OK" if ok else "FAIL"), q, "enough=", r["enough_evidence"], "raw_max=", round(r.get("raw_max_score") or 0, 3),
          "reason=", r.get("rejected_reason"))

report = {
    "n_positive": len(GOLD_POS),
    "positive_hit_rate": pos_hits / len(GOLD_POS),
    "n_negative": len(GOLD_NEG),
    "neg_ok_rate": neg_ok / len(GOLD_NEG),
    "k": k,
    "embedder": embedder.model_name,
    "min_score": retriever.min_score,
    "score_margin": retriever.score_margin,
    "bm25": bm25 is not None,
    "index_n": store.size,
}
print(json.dumps(report, ensure_ascii=False, indent=2))

eval_dir = PROJECT_ROOT / "eval" / "reports"
eval_dir.mkdir(parents=True, exist_ok=True)
out = eval_dir / "retrieval_smoke.json"
out.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("wrote", out)


OK Khám sàng lọc trước tiêm chủng enough= True n_hits= 5
OK Chống chỉ định tiêm enough= True n_hits= 3
OK Vaccine HPV phòng bệnh gì enough= True n_hits= 1
OK Trẻ sơ sinh tiêm TCMR enough= True n_hits= 1
FAIL Thời tiết Đà Nẵng ngày mai enough= True raw_max= 0.7 reason= None
FAIL Công thức nấu bún chả enough= True raw_max= 0.7 reason= None
FAIL Tỷ số bóng đá hôm qua enough= True raw_max= 0.7 reason= None
FAIL Cách học tiếng Anh giao tiếp enough= True raw_max= 0.7 reason= None
FAIL Giá vàng SJC enough= True raw_max= 0.7 reason= None
{
  "n_positive": 4,
  "positive_hit_rate": 1.0,
  "n_negative": 5,
  "neg_ok_rate": 0.0,
  "k": 5,
  "embedder": "BAAI/bge-m3",
  "min_score": 0.48,
  "score_margin": 0.08,
  "bm25": true,
  "index_n": 100
}
wrote d:\TTTN-VaxCare-2026\vaxcare-ai\eval\reports\retrieval_smoke.json
